In [ ]:
from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

DATA_DIR = Path(
    "/blue/bala1s/krishnap.kalivel/"
    "TLFHydrodynamics/thinfilm-closure-ml/"
    "datasets/BasC/param_space/ClosureData"
)

LAMBDA = 3.0

H_MEAN = 1.0
H_TOLERANCE = 0.1
OUTLET_EXCLUDE_FRACTION = 0.05
Q_ABS_MIN = 1e-10
ONLY_DISTURBED_POINTS = False

FLAT_H_TOL = 1e-3
FLAT_HX_TOL = 1e-4
FLAT_HXX_TOL = 1e-5

TRAIN_TIME_FRACTION = None

K = 4
SAMPLES_PER_CASE = 10000
N_CANDIDATES = 100

# Minimum separation between the two histories being compared.
TIME_EXCLUSION = 20.0

rng_uniqueness = np.random.default_rng(42)

files = sorted(DATA_DIR.glob("thinfilm_closure_data_*.h5"))

if not files:
    raise FileNotFoundError(f"No closure files found in {DATA_DIR}")

In [ ]:
def parse_case_name(filename):
    match = re.search(
        r"Re(?P<Re>\d+)_"
        r"Ca(?P<Ca>\d+_\d+)_"
        r"theta(?P<theta>\d+(?:_\d+)?)",
        Path(filename).stem
    )

    if match is None:
        raise ValueError(f"Cannot parse case parameters: {filename}")

    return {
        "Re": float(match["Re"]),
        "Ca": float(match["Ca"].replace("_", ".")),
        "theta": float(match["theta"].replace("_", "."))
    }


def load_case(filepath):
    field_names = [
        "h", "q", "shape_factor", "tau_wall",
        "h_x", "q_x", "h_xx", "q_xx",
        "h_m", "q_m", "shape_factor_m", "tau_wall_m",
    ]

    with h5py.File(filepath, "r") as f:
        missing = [
            name for name in ["x", "t", "tm", *field_names]
            if name not in f
        ]

        if missing:
            raise KeyError(f"{filepath.name}: missing datasets {missing}")

        x = np.asarray(f["x"][:], dtype=float).reshape(-1)
        t = np.asarray(f["t"][:], dtype=float).reshape(-1)
        tm = np.asarray(f["tm"][:], dtype=float).reshape(-1)

        fields = {
            name: np.asarray(f[name][:], dtype=float)
            for name in field_names
        }

    if tm.shape != t.shape:
        raise ValueError(
            f"{filepath.name}: t and tm must have the same length."
        )

    if (
        x.size < 3
        or not np.all(np.isfinite(x))
        or np.any(np.diff(x) <= 0)
    ):
        raise ValueError("x must contain at least 3 increasing finite points.")

    if not np.all(np.isfinite(t)) or np.any(np.diff(t) <= 0):
        raise ValueError("t must be finite and strictly increasing.")

    expected = (x.size, t.size)

    if x.size == t.size:
        raise ValueError(f"{filepath.name}: ambiguous spatial/time axes.")

    for name, array in fields.items():
        if array.shape == expected:
            pass
        elif array.shape == expected[::-1]:
            fields[name] = array.T
        else:
            raise ValueError(
                f"{filepath.name}: {name} has shape {array.shape}; "
                f"expected {expected} or {expected[::-1]}."
            )

    if TRAIN_TIME_FRACTION is not None:
        if not 0 < TRAIN_TIME_FRACTION <= 1:
            raise ValueError("TRAIN_TIME_FRACTION must lie in (0, 1].")

        nt_keep = int(np.floor(TRAIN_TIME_FRACTION * t.size))

        if nt_keep < 1:
            raise ValueError("Training selection contains no snapshots.")

        t = t[:nt_keep]
        tm = tm[:nt_keep]
        fields = {
            name: array[:, :nt_keep]
            for name, array in fields.items()
        }

    # Keep the existing three-value return signature.
    fields["tm"] = tm
    fields["dt"] = t - tm

    return x, t, fields

In [ ]:
def correction_terms(x, h, q, M, tau, h_x, Re):
    """Unmasked correction terms on arrays shaped (nx, nt)."""

    h_safe = np.where(
        np.isfinite(h) & (h > 0), h, np.nan
    )

    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        integral_difference = (
            M
            - q**2 / h_safe
            - (LAMBDA**2 / 45) * h_safe**5
        )

        C_M = np.gradient(
            integral_difference, x, axis=0, edge_order=2
        )

        C_tau = (
            (tau - 3 * q / h_safe**2) / Re
            + (LAMBDA**2 / 15) * h_safe**4 * h_x
        )

    return C_M, C_tau


def calculate_corrections(x, fields, Re):
    h = fields["h"]
    q = fields["q"]
    M = fields["shape_factor"]
    tau = fields["tau_wall"]

    # Stored Savitzky–Golay filtered derivatives.
    h_x = fields["h_x"]
    h_xx = fields["h_xx"]

    x_cutoff = (
        x[0] + (1 - OUTLET_EXCLUDE_FRACTION) * (x[-1] - x[0])
    )
    spatial_mask = x <= x_cutoff

    disturbed = (
        np.isfinite(h)
        & (np.abs(h - H_MEAN) > H_TOLERANCE)
        & spatial_mask[:, None]
    )

    if ONLY_DISTURBED_POINTS:
        analysis_mask = disturbed.copy()
    else:
        analysis_mask = np.zeros_like(h, dtype=bool)

        for j in range(h.shape[1]):
            indices = np.flatnonzero(disturbed[:, j])
            if indices.size:
                analysis_mask[:indices[-1] + 1, j] = True

    analysis_mask &= spatial_mask[:, None]

    C_M, C_tau = correction_terms(
        x, h, q, M, tau, h_x, Re
    )

    valid = (
        analysis_mask
        & np.isfinite(h)
        & (h > 0)
        & np.isfinite(q)
        & (np.abs(q) > Q_ABS_MIN)
        & np.isfinite(M)
        & np.isfinite(tau)
        & np.isfinite(C_M)
        & np.isfinite(C_tau)
    )

    flat_film = (
        np.isfinite(h)
        & np.isfinite(h_x)
        & np.isfinite(h_xx)
        & (np.abs(h - 1.0) <= FLAT_H_TOL)
        & (np.abs(h_x) <= FLAT_HX_TOL)
        & (np.abs(h_xx) <= FLAT_HXX_TOL)
    )

    valid &= ~flat_film  # Comment out to disable this extra filter.

    return {
        "momentum": np.where(valid, C_M, np.nan),
        "wall": np.where(valid, C_tau, np.nan)
    }


def calculate_previous_corrections(x, fields, Re):
    h_m = fields["h_m"]
    q_m = fields["q_m"]
    M_m = fields["shape_factor_m"]
    tau_m = fields["tau_wall_m"]

    h_m_safe = np.where(
        np.isfinite(h_m) & (h_m > 0), h_m, np.nan
    )

    # No stored h_x_m: construct it spatially.
    h_x_m = np.gradient(
        h_m_safe, x, axis=0, edge_order=2
    )

    C_M_m, C_tau_m = correction_terms(
        x, h_m, q_m, M_m, tau_m, h_x_m, Re
    )

    valid_m = (
        np.isfinite(h_m)
        & (h_m > 0)
        & np.isfinite(q_m)
        & np.isfinite(M_m)
        & np.isfinite(tau_m)
        & np.isfinite(h_x_m)
        & np.isfinite(C_M_m)
        & np.isfinite(C_tau_m)
    )

    # Do not exclude a previous flat state: it can be valid history.
    return {
        "momentum": np.where(valid_m, C_M_m, np.nan),
        "wall": np.where(valid_m, C_tau_m, np.nan)
    }

In [ ]:
window_parts = []
local_parts = []
memory_parts = []
target_parts = []
metadata_parts = []

for case_id, filepath in enumerate(files):
    parameters = parse_case_name(filepath.name)
    x, t, fields = load_case(filepath)

    if x.size < 2 * K + 1:
        print(f"Skipping {filepath.stem}: insufficient spatial points.")
        continue

    h = fields["h"]
    q = fields["q"]

    h_x = fields["h_x"]
    q_x = fields["q_x"]
    h_xx = fields["h_xx"]
    q_xx = fields["q_xx"]

    tm = fields["tm"]
    dt = fields["dt"]

    current = calculate_corrections(x, fields, parameters["Re"])
    previous = calculate_previous_corrections(
        x, fields, parameters["Re"]
    )

    valid_time = (
        np.isfinite(tm)
        & np.isfinite(dt)
        & (dt > 0)
    )

    valid = (
        np.isfinite(current["momentum"])
        & np.isfinite(current["wall"])
        & np.isfinite(h_x)
        & np.isfinite(q_x)
        & np.isfinite(h_xx)
        & np.isfinite(q_xx)
        & np.isfinite(fields["h_m"])
        & np.isfinite(fields["q_m"])
        & np.isfinite(previous["momentum"])
        & np.isfinite(previous["wall"])
        & valid_time[None, :]
    )

    valid[:K, :] = False
    valid[-K:, :] = False

    # Require finite h and q throughout each current window.
    for offset in range(-K, K + 1):
        valid[K:-K, :] &= (
            np.isfinite(h[K + offset:x.size - K + offset, :])
            & np.isfinite(q[K + offset:x.size - K + offset, :])
        )

    ix, it = np.where(valid)

    if ix.size == 0:
        print(f"Skipping {filepath.stem}: no valid paired samples.")
        continue

    if ix.size > SAMPLES_PER_CASE:
        chosen = rng_uniqueness.choice(
            ix.size, SAMPLES_PER_CASE, replace=False
        )
        ix, it = ix[chosen], it[chosen]

    params = np.tile(
        [parameters["Re"], parameters["Ca"], parameters["theta"]],
        (ix.size, 1)
    )

    X_window_case = np.column_stack(
        [h[ix + offset, it] for offset in range(-K, K + 1)]
        + [q[ix + offset, it] for offset in range(-K, K + 1)]
        + [params]
    )

    X_local_case = np.column_stack([
        h[ix, it],
        q[ix, it],
        h_x[ix, it],
        q_x[ix, it],
        h_xx[ix, it],
        q_xx[ix, it],
        params,
    ])

    X_memory_case = np.column_stack([
        X_local_case,
        fields["h_m"][ix, it],
        fields["q_m"][ix, it],
        previous["momentum"][ix, it],
        previous["wall"][ix, it],
        dt[it],
    ])

    Y_case = np.column_stack([
        current["momentum"][ix, it],
        current["wall"][ix, it],
    ])

    window_parts.append(X_window_case)
    local_parts.append(X_local_case)
    memory_parts.append(X_memory_case)
    target_parts.append(Y_case)

    metadata_parts.append(pd.DataFrame({
        "case_id": case_id,
        "case": filepath.stem,
        "t": t[it],
        "tm": tm[it],
        "dt": dt[it],
        "x": x[ix],
        "ix": ix,
        "it": it,
    }))

    print(
        f"{filepath.stem}: {ix.size} samples"
        f" | dt range: {dt[valid_time].min():.6g}"
        f" to {dt[valid_time].max():.6g}"
    )

if not target_parts:
    raise ValueError("No valid samples found.")

X_window = np.vstack(window_parts)
X_local = np.vstack(local_parts)
X_memory = np.vstack(memory_parts)
Y = np.vstack(target_parts)

sample_metadata = pd.concat(metadata_parts, ignore_index=True)

print("\nWindow inputs:", X_window.shape)
print("Local inputs: ", X_local.shape)
print("Memory inputs:", X_memory.shape)
print("Corrections:  ", Y.shape)

In [ ]:
y_scale = Y.std(axis=0)
y_scale = np.where(y_scale > 0, y_scale, 1.0)


def neighbor_diagnostic(X, representation):
    X_scaled = StandardScaler().fit_transform(X)

    # Normalize by the number of nonconstant features.
    n_active = np.count_nonzero(np.std(X, axis=0) > 0)
    X_scaled /= np.sqrt(max(n_active, 1))

    rows = []
    n_without_neighbor = 0

    for case_id, group in sample_metadata.groupby("case_id"):
        indices = group.index.to_numpy()

        if indices.size < 2:
            continue

        model = NearestNeighbors(
            n_neighbors=min(N_CANDIDATES + 1, indices.size),
            n_jobs=-1
        ).fit(X_scaled[indices])

        distances, neighbors = model.kneighbors(X_scaled[indices])

        times = group["t"].to_numpy()
        previous_times = group["tm"].to_numpy()

        for position, sample_id in enumerate(indices):
            candidates = neighbors[position]

            # Positive gap only when the history intervals do not overlap.
            history_gap = np.maximum(
                previous_times[candidates] - times[position],
                previous_times[position] - times[candidates]
            )

            eligible = (
                (candidates != position)
                & (history_gap > TIME_EXCLUSION)
            )

            locations = np.flatnonzero(eligible)

            if locations.size == 0:
                n_without_neighbor += 1
                continue

            first = locations[0]
            neighbor_id = indices[candidates[first]]

            delta = np.abs(Y[sample_id] - Y[neighbor_id])

            rows.append({
                "representation": representation,
                "sample_id": sample_id,
                "neighbor_id": neighbor_id,
                "case": sample_metadata.loc[sample_id, "case"],
                "input_distance": distances[position, first],
                "momentum_difference": delta[0],
                "wall_difference": delta[1],
                "momentum_difference_scaled": delta[0] / y_scale[0],
                "wall_difference_scaled": delta[1] / y_scale[1],
            })

    print(
        f"{representation}: {len(rows)} pairs"
        f" | {n_without_neighbor} samples without an eligible "
        f"neighbor among searched candidates"
    )

    return pd.DataFrame(rows)


neighbor_results = pd.concat([
    neighbor_diagnostic(X_window, "Spatial window"),
    neighbor_diagnostic(X_local, "Local derivatives"),
    neighbor_diagnostic(X_memory, "Local + memory"),
], ignore_index=True)

if neighbor_results.empty:
    raise ValueError(
        "No eligible neighbors. Increase N_CANDIDATES or "
        "reduce TIME_EXCLUSION."
    )

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(18, 9), sharey="row"
)

representations = [
    "Spatial window",
    "Local derivatives",
    "Local + memory",
]

for col, representation in enumerate(representations):
    df = neighbor_results[
        neighbor_results["representation"] == representation
    ]

    for row, quantity in enumerate(["momentum", "wall"]):
        ax = axes[row, col]

        ax.scatter(
            df["input_distance"],
            df[f"{quantity}_difference_scaled"],
            s=8, alpha=0.2
        )

        ax.set_ylabel(
            f"|Difference in {quantity} correction| / std"
        )
        ax.grid(alpha=0.25)
        ax.set_xlim(0, 1.5)
        ax.set_ylim(0, 10)

    axes[0, col].set_title(representation)
    axes[-1, col].set_xlabel("Standardized input distance")

    

plt.tight_layout()
plt.show()

Just one simulation

In [ ]:
RE = 120
CA = 0.01
THETA = 45

K = 4
TIME_EXCLUSION = 20.0
QUERY_BATCH_SIZE = 1000

re_tag = f"{RE:03d}"
ca_tag = f"{CA:.3f}".replace(".", "_")
theta_tag = f"{THETA:g}".replace(".", "_")

filename = (
    f"thinfilm_closure_data_Re{re_tag}_"
    f"Ca{ca_tag}_theta{theta_tag}.h5"
)
filepath = DATA_DIR / filename

if not filepath.is_file():
    raise FileNotFoundError(filepath)

# Use all stored snapshots for this analysis.
previous_fraction = TRAIN_TIME_FRACTION
try:
    TRAIN_TIME_FRACTION = None
    x, t, fields = load_case(filepath)
finally:
    TRAIN_TIME_FRACTION = previous_fraction

current = calculate_corrections(x, fields, RE)
previous = calculate_previous_corrections(x, fields, RE)

print(filename)
print(f"Time range: {t[0]:g} to {t[-1]:g}")

In [ ]:
h = fields["h"]
q = fields["q"]
tm = fields["tm"]
dt = fields["dt"]

if x.size < 2 * K + 1:
    raise ValueError("Insufficient spatial points for the chosen K.")

valid_time = np.isfinite(tm) & np.isfinite(dt) & (dt > 0)

valid = (
    np.isfinite(current["momentum"])
    & np.isfinite(current["wall"])
    & np.isfinite(fields["h_x"])
    & np.isfinite(fields["q_x"])
    & np.isfinite(fields["h_xx"])
    & np.isfinite(fields["q_xx"])
    & np.isfinite(fields["h_m"])
    & np.isfinite(fields["q_m"])
    & np.isfinite(previous["momentum"])
    & np.isfinite(previous["wall"])
    & valid_time[None, :]
)

valid[:K, :] = False
valid[-K:, :] = False

for offset in range(-K, K + 1):
    valid[K:-K, :] &= (
        np.isfinite(h[K + offset:x.size - K + offset, :])
        & np.isfinite(q[K + offset:x.size - K + offset, :])
    )

ix, it = np.where(valid)

if ix.size < 2:
    raise ValueError("Insufficient valid samples.")

# All valid samples are retained.
X_window_case = np.column_stack(
    [h[ix + offset, it] for offset in range(-K, K + 1)]
    + [q[ix + offset, it] for offset in range(-K, K + 1)]
)

X_local_case = np.column_stack([
    h[ix, it],
    q[ix, it],
    fields["h_x"][ix, it],
    fields["q_x"][ix, it],
    fields["h_xx"][ix, it],
    fields["q_xx"][ix, it],
])

X_memory_case = np.column_stack([
    X_local_case,
    fields["h_m"][ix, it],
    fields["q_m"][ix, it],
    previous["momentum"][ix, it],
    previous["wall"][ix, it],
    dt[it],
])

Y_case = np.column_stack([
    current["momentum"][ix, it],
    current["wall"][ix, it],
])

metadata_case = pd.DataFrame({
    "ix": ix,
    "it": it,
    "x": x[ix],
    "t": t[it],
    "tm": tm[it],
    "dt": dt[it],
})

print(f"Retained {ix.size:,} samples — no downsampling.")
print("Window:", X_window_case.shape)
print("Local: ", X_local_case.shape)
print("Memory:", X_memory_case.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors


def single_case_diagnostic(X, Y, metadata, representation):
    n = X.shape[0]

    # Remove constant features, including dt if constant in this case.
    active = np.std(X, axis=0) > 0
    if not np.any(active):
        raise ValueError(f"{representation}: all input features are constant.")

    Z = StandardScaler().fit_transform(X[:, active])
    Z /= np.sqrt(np.count_nonzero(active))

    output_scale = Y.std(axis=0)
    output_scale = np.where(output_scale > 0, output_scale, 1.0)

    times = metadata["t"].to_numpy()
    previous_times = metadata["tm"].to_numpy()

    # Check whether any eligible time interval exists for each query.
    can_match = (
        (previous_times.max() - times > TIME_EXCLUSION)
        | (previous_times - times.min() > TIME_EXCLUSION)
    )

    model = NearestNeighbors(
        algorithm="auto", n_jobs=-1
    ).fit(Z)

    neighbor_ids = np.full(n, -1, dtype=int)
    input_distances = np.full(n, np.nan)

    for start in range(0, n, QUERY_BATCH_SIZE):
        stop = min(start + QUERY_BATCH_SIZE, n)
        pending = np.arange(start, stop)
        pending = pending[can_match[pending]]

        n_search = min(64, n)

        while pending.size:
            distances, candidates = model.kneighbors(
                Z[pending], n_neighbors=n_search
            )

            history_gap = np.maximum(
                previous_times[candidates] - times[pending, None],
                previous_times[pending, None] - times[candidates]
            )

            eligible = (
                (candidates != pending[:, None])
                & (history_gap > TIME_EXCLUSION)
            )

            found = eligible.any(axis=1)
            first = eligible.argmax(axis=1)
            rows = np.flatnonzero(found)

            query_ids = pending[found]
            neighbor_ids[query_ids] = candidates[rows, first[found]]
            input_distances[query_ids] = distances[rows, first[found]]

            pending = pending[~found]

            if not pending.size or n_search == n:
                break

            n_search = min(2 * n_search, n)

        print(
            f"{representation}: processed {stop:,}/{n:,}",
            end="\r"
        )

    sample_ids = np.flatnonzero(neighbor_ids >= 0)
    selected_neighbors = neighbor_ids[sample_ids]

    difference = np.abs(
        Y[sample_ids] - Y[selected_neighbors]
    )

    print(
        f"\n{representation}: {sample_ids.size:,} eligible pairs; "
        f"{n - sample_ids.size:,} samples without eligible neighbors."
    )

    return pd.DataFrame({
        "representation": representation,
        "sample_id": sample_ids,
        "neighbor_id": selected_neighbors,
        "input_distance": input_distances[sample_ids],
        "momentum_difference": difference[:, 0],
        "wall_difference": difference[:, 1],
        "momentum_difference_scaled": difference[:, 0] / output_scale[0],
        "wall_difference_scaled": difference[:, 1] / output_scale[1],
    })


case_neighbor_results = pd.concat([
    single_case_diagnostic(
        X_window_case, Y_case, metadata_case, "Spatial window"
    ),
    single_case_diagnostic(
        X_local_case, Y_case, metadata_case, "Local derivatives"
    ),
    single_case_diagnostic(
        X_memory_case, Y_case, metadata_case, "Local + memory"
    ),
], ignore_index=True)

if case_neighbor_results.empty:
    raise ValueError("No eligible pairs; check TIME_EXCLUSION.")

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(18, 9), sharey="row"
)

representations = [
    "Spatial window",
    "Local derivatives",
    "Local + memory",
]

for col, representation in enumerate(representations):
    df = case_neighbor_results[
        case_neighbor_results["representation"] == representation
    ]

    for row, quantity in enumerate(["momentum", "wall"]):
        ax = axes[row, col]

        ax.scatter(
            df["input_distance"],
            df[f"{quantity}_difference_scaled"],
            s=2, alpha=0.15, rasterized=True
        )

        ax.set_ylabel(
            f"|Difference in {quantity} correction| / case std"
        )
        ax.grid(alpha=0.25)

    axes[0, col].set_title(representation)
    axes[-1, col].set_xlabel("Standardized input distance")

fig.suptitle(f"Re={RE:g}, Ca={CA:g}, θ={THETA:g}°")
plt.tight_layout()
plt.show()